# 01 - Method agreement
European prices from BSM, CRR, Leisen-Reimer, Crank-Nicolson and Monte Carlo, plus American options (trees and PDE only). **Inputs are model parameters, not market data.**

In [1]:
%matplotlib inline
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from optpricer import Option, MarketData, price, price_all, bsm
pd.set_option("display.width", 160); pd.set_option("display.precision", 6)

In [2]:
m = MarketData(S=100, r=0.05, q=0.0, sigma=0.20)
for kind in ("call", "put"):
    df = price_all(Option(kind, 100, 1.0), m, methods=["bs", "crr", "lr", "cn", "mc"],
                   settings={"crr": {"n": 1000}, "lr": {"n": 201}, "mc": {"n_paths": 200_000, "seed": 1}})
    df["diff_vs_bs"] = df["price"] - df.loc[df.method == "bs", "price"].iloc[0]
    display(df[["method", "price", "std_error", "diff_vs_bs", "runtime_s", "settings"]])

,method,price,std_error,diff_vs_bs,runtime_s,settings
0,bs,10.450584,NaN,0.000000,0.000468,{}
1,crr,10.448584,NaN,-0.001999,0.003994,{'n': 1000}
2,lr,10.450575,NaN,-0.000009,0.000625,{'n': 201}
3,cn,10.450713,NaN,0.000130,0.011551,"{'n_space': 400, 'n_time': 400}"
4,mc,10.399278,0.032837,-0.051306,0.006775,"{'n_paths': 200000, 'antithetic': False, 'seed..."


,method,price,std_error,diff_vs_bs,runtime_s,settings
0,bs,5.573526,NaN,0.000000,0.000373,{}
1,crr,5.571527,NaN,-0.001999,0.004011,{'n': 1000}
2,lr,5.573517,NaN,-0.000009,0.000603,{'n': 201}
3,cn,5.573474,NaN,-0.000052,0.011463,"{'n_space': 400, 'n_time': 400}"
4,mc,5.576911,0.019374,0.003385,0.006709,"{'n_paths': 200000, 'antithetic': False, 'seed..."


Deterministic methods differ from BSM by discretisation error. Monte Carlo differs by sampling error and should be judged against its standard error, never used to excuse tree/PDE bias.

In [3]:
am = Option("put", 100, 1.0, "american")
display(price_all(am, m, methods=["bs", "mc", "crr", "lr", "cn"], settings={"lr": {"n": 1001}})[["method", "price", "status"]])
res = price(am, m, "cn", n_space=800, n_time=800)
{k: v for k, v in res.diagnostics.items() if k.startswith(("psor", "complementarity", "obstacle"))}

,method,price,status
0,bs,NaN,unsupported for american
1,mc,NaN,unsupported for american
2,crr,6.089595,ok
3,lr,6.090082,ok
4,cn,6.089759,ok


{'psor_omega': 1.2,
 'psor_tol': 1e-10,
 'psor_max_iter': 10000,
 'psor_converged': True,
 'psor_total_iterations': 10566,
 'psor_max_iterations_per_step': 15,
 'psor_mean_iterations_per_step': 13.174563591022444,
 'obstacle_violation_max': 0.0,
 'complementarity_residual_max': 7.905653909290322e-11}

BSM and terminal-payoff MC are refused for American options. The PSOR diagnostics report iterations, convergence and the complementarity residual of the linear complementarity problem.

In [4]:
from optpricer.binomial import leisen_reimer
for r in (0.05, 0.0, -0.02):
    mm = MarketData(130, r, 0.0, 0.2)
    a = leisen_reimer(Option("call", 100, 2.0, "american"), mm, 1001).price
    e = leisen_reimer(Option("call", 100, 2.0), mm, 1001).price
    print(f"q=0, r={r:+.2f}: American - European call = {a - e:.3e}")

q=0, r=+0.05: American - European call = 0.000e+00
q=0, r=+0.00: American - European call = 4.121e-13
q=0, r=-0.02: American - European call = 1.249e+00


With q = 0 the American call equals the European call when r >= 0. At a negative rate early exercise of a deep in-the-money call can be optimal, so the equality fails.